# 🎬 AI Video Dubber v6.0 — Industry-Standard Pipeline

**Professional AI dubbing** — translates and dubs videos into 18+ languages with:

| Feature | What It Does |
|---------|-------------|
| 🎬 **Director v4** | Chunked analysis, pitch-based speaker detection, scene boundaries |
| 🌐 **Two-Pass Translation** | Draft → Polish for coherent, natural dialogue |
| ✍️ **Voice Bible Writer** | Character-consistent dialogue with pronunciation hints |
| 🎭 **SSML Voice Casting** | Distinct voices per character (father sounds different from son!) |
| ⏱️ **Timestamp Aligner** | Rubberband time-stretch for perfect lip sync |
| 🎵 **Demucs** | Removes original vocals, keeps clean background |
| 🔊 **Per-clip Normalization** | EBU R128 loudness on every clip + final mix |
| 🐟 **Fish Speech** (optional) | Local GPU TTS for premium quality |

### ⚡ Requirements
- **T4 GPU**: Runtime → Change runtime type → T4 GPU
- **Groq API Key** (FREE): https://console.groq.com/keys

---

## Step 1: Setup Environment & GPU

In [ ]:
# ⚡ First: Enable GPU → Runtime → Change runtime type → T4 GPU

import subprocess, os

# Check GPU
gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f'✅ GPU: {gpu_check.stdout.strip()}')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/DubbedVideos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f'✅ Google Drive mounted! Output: {DRIVE_OUTPUT}')

## Step 2: Install Dependencies

In [ ]:
# Install core dependencies
!pip install -q groq>=0.11.0 edge-tts>=6.1.0 yt-dlp indic-transliteration>=2.3.0 nest-asyncio>=1.5.0
!pip install -q demucs httpx torchaudio

# Apply nest_asyncio fix for Colab
import nest_asyncio
nest_asyncio.apply()

# Verify
!ffmpeg -version 2>&1 | head -1
!yt-dlp --version
print('\n✅ Core dependencies installed!')

## Step 3: Install Fish Speech (Optional — LOCAL GPU TTS)

Fish Speech runs **directly on the Colab T4 GPU**. No API key, no cost.

**Skip this step** if you just want to use Edge TTS (which still produces good quality with v6.0's SSML prosody system).

In [ ]:
%%time
import os, subprocess, time

FISH_DIR = '/content/fish-speech'
MODEL_DIR = '/content/fish-speech-model'

# Step 1: Clone Fish Speech
if not os.path.exists(FISH_DIR):
    print('📦 Cloning Fish Speech...')
    !git clone https://github.com/fishaudio/fish-speech.git {FISH_DIR}
else:
    print('✅ Fish Speech already cloned')

# Step 2: Install Fish Speech
print('📦 Installing Fish Speech...')
!cd {FISH_DIR} && pip install -q -e . 2>&1 | tail -3

# Step 3: Download model
if not os.path.exists(MODEL_DIR):
    print('📥 Downloading Fish Speech model (~2GB)...')
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='fishaudio/fish-speech-1.5',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md', '*.txt', 'samples/*']
    )
    print(f'✅ Model downloaded to {MODEL_DIR}')
else:
    print('✅ Model already downloaded')

print('\n✅ Fish Speech ready for local GPU inference!')

In [ ]:
# Start Fish Speech local API server in the background
import subprocess, time, os

FISH_DIR = '/content/fish-speech'
MODEL_DIR = '/content/fish-speech-model'

# Kill any existing server
!pkill -f 'fish_speech' 2>/dev/null || true
time.sleep(1)

print('🐟 Starting Fish Speech local server on port 8080...')

api_server_path = os.path.join(FISH_DIR, 'tools', 'api_server.py')
if os.path.exists(api_server_path):
    server_cmd = f'cd {FISH_DIR} && python tools/api_server.py --listen 0.0.0.0 --port 8080 --checkpoint-path {MODEL_DIR}'
else:
    server_cmd = f'cd {FISH_DIR} && python -m fish_speech.serve --listen 0.0.0.0 --port 8080 --checkpoint-path {MODEL_DIR}'

server_proc = subprocess.Popen(
    server_cmd, shell=True,
    stdout=open('/content/fish_server.log', 'w'),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)

print('⏳ Waiting for server to load model...')
import httpx
for i in range(60):
    time.sleep(2)
    try:
        r = httpx.get('http://localhost:8080/', timeout=2.0)
        if r.status_code < 500:
            print(f'\n✅ Fish Speech server running on localhost:8080!')
            break
    except:
        if i % 5 == 0:
            print(f'   Loading... ({i*2}s)')
else:
    print('\n⚠️ Fish Speech did not start. Pipeline will use Edge TTS (still excellent with v6.0).')
    !tail -10 /content/fish_server.log

## Step 4: Clone Dubber Agent & Set Groq Key

In [ ]:
# Clone the dubber repo
REPO_DIR = '/content/chinese-drama-dubber'
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR)
!git clone https://github.com/Dipesh600/chinese-drama-dubber.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'📁 Working directory: {os.getcwd()}')
print(f'\n✅ Repository cloned!')

In [ ]:
# Set Groq API Key (the ONLY key you need — it's free)
# Get yours at: https://console.groq.com/keys

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key loaded from Colab Secrets!')
except:
    import getpass
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter your Groq API Key: ')
    print('✅ Groq API key set!')

key = os.environ.get('GROQ_API_KEY', '')
print(f'🔑 Groq Key: {key[:8]}...{key[-4:]}' if len(key) > 12 else '❌ No key set!')

# Check Fish Speech status
try:
    import httpx
    r = httpx.get('http://localhost:8080/', timeout=2.0)
    print(f'🐟 Fish Speech: ✅ Running locally on GPU')
except:
    print(f'🔊 TTS: Edge TTS with SSML prosody (character differentiation)')

## Step 5: 🎬 Dub Your Video!

### Supported Languages
Hindi, Nepali, Tamil, Telugu, Bengali, Marathi, Gujarati, Kannada, Malayalam, Urdu, English, Spanish, French, Portuguese, German, Japanese, Korean, Arabic, Turkish

In [ ]:
#@title 🎬 Dubbing Configuration { display-mode: "form" }

#@markdown ### Video Settings
VIDEO_URL = 'https://youtu.be/m_mJhZuM4Bk' #@param {type:"string"}
TARGET_LANGUAGE = 'Hindi' #@param ['Hindi', 'Nepali', 'Tamil', 'Telugu', 'Bengali', 'Marathi', 'Gujarati', 'Kannada', 'Malayalam', 'Urdu', 'English', 'Spanish', 'French', 'Portuguese', 'German', 'Japanese', 'Korean', 'Arabic', 'Turkish']
SOURCE_LANGUAGE = 'zh' #@param ['zh', 'en', 'ja', 'ko', 'auto']

#@markdown ### Audio Settings
KEEP_BACKGROUND_MUSIC = True #@param {type:"boolean"}
USE_DEMUCS = True #@param {type:"boolean"}

#@markdown ### Description
VIDEO_DESCRIPTION = 'Chinese drama video' #@param {type:"string"}

print(f'🎬 Video:      {VIDEO_URL}')
print(f'🌐 Language:   {TARGET_LANGUAGE}')
print(f'🎵 BG Music:   {"Demucs + Smart Ducking" if USE_DEMUCS else "Smart Ducking"}')

# Check TTS engine
try:
    import httpx
    r = httpx.get('http://localhost:8080/', timeout=2.0)
    tts_info = '🐟 Fish Speech LOCAL (GPU) + Edge TTS SSML fallback'
except:
    tts_info = '🔊 Edge TTS with SSML prosody (character differentiation)'
print(f'🔊 TTS:        {tts_info}')

print(f'📁 Output:     {DRIVE_OUTPUT}')
print(f'\n⏳ Starting v6.0 industry-standard pipeline...')
print('=' * 70)

In [ ]:
# Run the v6.0 industry-standard dubbing pipeline!
import sys, importlib
sys.path.insert(0, REPO_DIR)

# Force reload all modules
for mod_name in list(sys.modules.keys()):
    if mod_name in ['orchestrator', 'director', 'preprocessor', 'translator',
                    'dialogue_writer', 'sentence_merger', 'voice_caster',
                    'voice_catalog', 'tts_engine', 'assembler', 'romanizer',
                    'validator', 'audio_separator', 'timestamp_aligner']:
        del sys.modules[mod_name]

from orchestrator import run_agent

result = run_agent(
    url=VIDEO_URL,
    target_lang=TARGET_LANGUAGE,
    source_lang=SOURCE_LANGUAGE,
    user_description=VIDEO_DESCRIPTION,
    output_dir=DRIVE_OUTPUT,
    preserve_bg=KEEP_BACKGROUND_MUSIC,
    use_demucs=USE_DEMUCS
)

if result['success']:
    print('\n' + '=' * 70)
    print(f'  ✅ DUBBING COMPLETE!')
    print(f'  📹 Video:      {result["video_path"]}')
    print(f'  📝 Subtitles:  {result["srt_path"]}')
    print(f'  🎭 Content:    {result.get("content_type", "?")} | {result.get("real_speaker_count", "?")} speakers')
    print(f'  🎬 Scenes:     {result.get("scenes", "?")}')
    print(f'  ⏱️ Time:       {result.get("processing_time", "?")}s')
    print(f'  💾 Size:       {result.get("size_mb", "?")}MB')
    print(f'  🗂️ Saved to:   Google Drive → DubbedVideos/')
    print('=' * 70)
else:
    print(f'\n❌ FAILED at stage {result.get("stage", "?")}: {result.get("error", "Unknown")}')
    print(f'   Work dir: {result.get("work_dir", "?")}')
    print(f'\n💡 Pipeline has crash recovery — just re-run this cell!')

## Step 6: Preview & Download

In [ ]:
# Play the dubbed video!
if result.get('success'):
    from IPython.display import Video, display
    try:
        display(Video(result['video_path'], embed=True, width=640))
    except:
        print(f'Video at: {result["video_path"]}')

    srt_path = result.get('srt_path', '')
    if srt_path and os.path.exists(srt_path):
        print('\n📝 Subtitles:')
        with open(srt_path, encoding='utf-8') as f:
            print(''.join(f.readlines()[:15]))
else:
    print('No video to preview.')

In [ ]:
# Save clean copy to Google Drive
if result.get('success'):
    import shutil, time
    ts = time.strftime('%Y%m%d_%H%M%S')
    final_dir = os.path.join(DRIVE_OUTPUT, f'dubbed_{TARGET_LANGUAGE}_{ts}')
    os.makedirs(final_dir, exist_ok=True)

    shutil.copy2(result['video_path'], os.path.join(final_dir, f'dubbed_{TARGET_LANGUAGE}.mp4'))
    if os.path.exists(result.get('srt_path', '')):
        shutil.copy2(result['srt_path'], os.path.join(final_dir, f'subtitles_{TARGET_LANGUAGE}.srt'))

    print(f'✅ Saved to: Google Drive → DubbedVideos → dubbed_{TARGET_LANGUAGE}_{ts}/')

---
## 🔄 Dub Another Video
Go back to **Step 5**, change URL/language, and run again!

### 💡 Tips
- **T4 GPU recommended** for Fish Speech + Demucs (but Edge TTS works on CPU too)
- **Only 1 API key needed**: Groq (free at console.groq.com)
- **Crash recovery**: If pipeline fails, just re-run — it resumes from where it stopped
- **Fish Speech** runs locally on GPU — zero cost premium TTS
- **Edge TTS** with v6.0's SSML prosody system produces distinct voices per character even without Fish Speech